# Smoke Test — April 9 + Both April 20 Records

Runs all three record trainers back-to-back on a single Colab GPU using the same data.

**What each cell does:**
1. GPU check
2. Clone / update repo
3. Install repo + notebook dependencies
4. Download shared data and build an explicit capped smoke-val split (used by all three models)
5. Train + export: April 9 `LegalTTT`
6. Train + export: April 20 `AttnGate_MultiPhaseTTT_LaCT`
7. Train + export: April 20 `3LayerRecur_ParResid_AttnGate_MP4TTT`
8. Compare final metrics from all three logs

**Smoke-test settings (all three models share these):**
- `ITERATIONS=300` — fast check; not a quality run
- `TRAIN_BATCH_TOKENS=32768` — single-GPU batch (full record uses 786432 on 8×H100)
- `SMOKE_VAL_MAX_TOKENS=4194304` — explicit capped validation prefix shared by all three runs
- `VAL_BATCH_TOKENS=32768` — reduced eval batch size on that capped validation set
- `MAX_WALLCLOCK_SECONDS=0` — no time cap; let all 300 steps finish
- `VAL_LOSS_EVERY=0` — skip periodic val; final only
- `GPTQ_CALIBRATION_BATCHES=8` — fast export check
- `PG_COLAB_DISABLE_COMPILE=1` — disables `torch.compile` via `sitecustomize.py`
- No FlashAttention source build in this notebook; the local `flash_attn_interface.py` stub is used where needed

**Why `4194304` val tokens:** this is `2048` full eval sequences of length `2048` and `128` legal-TTT chunks of `32768` tokens, so each model still gets millions of scored tokens and a meaningful adaptation schedule instead of just a handful of updates. It is also about `10.3%` of the April 9 full validation load (`40540160` tokens), which should cut sliding-window and TTT smoke-eval time by roughly `9.7x` while keeping the three smoke runs directly comparable to each other.

**OOM recovery:** lower `GPTQ_CALIBRATION_BATCHES=4` or `TRAIN_BATCH_TOKENS=16384` in the failing cell only.

In [ ]:
%%bash
nvidia-smi

In [ ]:
%%bash
set -euo pipefail
REPO_DIR="$(pwd)/parameter-golf"
if [[ ! -d "$REPO_DIR/.git" ]]; then
  git clone https://github.com/IanniMuliterno/parameter-golf.git "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin
  git -C "$REPO_DIR" pull --ff-only
fi
echo "commit: $(git -C "$REPO_DIR" rev-parse --short HEAD)"

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
python3 -m pip install -r "$REPO/requirements.txt" -r "$REPO/colab/2026-04-23_SmokeTest_Records/requirements.txt"
python3 - <<'PY'
import brotli
import huggingface_hub
import sentencepiece
print('Notebook dependencies ready.')
PY

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
# Download 5 train shards + full val, then build one explicit capped smoke-val root.
# sp8192 dataset lives on kevclark/parameter-golf, not the default repo.
MATCHED_FINEWEB_REPO_ID=kevclark/parameter-golf \
  python3 "$REPO/data/cached_challenge_fineweb.py" --variant sp8192 --train-shards 5
python3 - <<'PY'
import os
import shutil
from pathlib import Path

import numpy as np

HEADER_INTS = 256
HEADER_BYTES = HEADER_INTS * np.dtype('<i4').itemsize
TOKEN_DTYPE = np.dtype('<u2')
EXPECTED_MAGIC = 20240520
EXPECTED_VERSION = 1
SMOKE_VAL_MAX_TOKENS = 4_194_304
EVAL_SEQ_LEN = 2048
TTT_CHUNK_TOKENS = 32_768

repo = Path('parameter-golf')
source_dataset_dir = repo / 'data' / 'datasets' / 'fineweb10B_sp8192'
source_tokenizer_dir = repo / 'data' / 'tokenizers'
smoke_data_dir = repo / 'data_smoke'
smoke_dataset_dir = smoke_data_dir / 'datasets' / 'fineweb10B_sp8192'
smoke_tokenizer_dir = smoke_data_dir / 'tokenizers'
train_files = sorted(source_dataset_dir.glob('fineweb_train_*.bin'))
val_files = sorted(source_dataset_dir.glob('fineweb_val_*.bin'))
if len(train_files) < 5:
    raise SystemExit(f'expected at least 5 train shards in {source_dataset_dir}, found {len(train_files)}')
if not val_files:
    raise SystemExit(f'expected at least 1 val shard in {source_dataset_dir}, found 0')
if SMOKE_VAL_MAX_TOKENS % EVAL_SEQ_LEN != 0:
    raise SystemExit(
        f'SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS} must be divisible by EVAL_SEQ_LEN={EVAL_SEQ_LEN}'
    )
if SMOKE_VAL_MAX_TOKENS % TTT_CHUNK_TOKENS != 0:
    raise SystemExit(
        f'SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS} must be divisible by TTT_CHUNK_TOKENS={TTT_CHUNK_TOKENS}'
    )

def read_header(path: Path) -> np.ndarray:
    header = np.fromfile(path, dtype='<i4', count=HEADER_INTS)
    if header.size != HEADER_INTS:
        raise SystemExit(f'short header in {path}')
    if int(header[0]) != EXPECTED_MAGIC or int(header[1]) != EXPECTED_VERSION:
        raise SystemExit(f'unexpected shard header in {path}: {header[:3].tolist()}')
    return header

full_val_tokens = 0
val_parts = []
remaining = SMOKE_VAL_MAX_TOKENS
first_header = None
for path in val_files:
    header = read_header(path)
    if first_header is None:
        first_header = header.copy()
    shard_tokens = int(header[2])
    full_val_tokens += shard_tokens
    if remaining <= 0:
        continue
    take = min(remaining, shard_tokens)
    tokens = np.fromfile(path, dtype=TOKEN_DTYPE, count=take, offset=HEADER_BYTES)
    if tokens.size != take:
        raise SystemExit(f'short token read in {path}: expected {take}, found {tokens.size}')
    val_parts.append(tokens)
    remaining -= take
if first_header is None:
    raise SystemExit('missing validation header after listing val shards')
if full_val_tokens < SMOKE_VAL_MAX_TOKENS:
    raise SystemExit(
        f'full validation split has only {full_val_tokens} tokens, below requested SMOKE_VAL_MAX_TOKENS={SMOKE_VAL_MAX_TOKENS}'
    )
if remaining != 0:
    raise SystemExit(f'failed to collect capped val tokens exactly; remaining={remaining}')

if smoke_data_dir.exists():
    shutil.rmtree(smoke_data_dir)
smoke_dataset_dir.mkdir(parents=True, exist_ok=False)
smoke_tokenizer_dir.mkdir(parents=True, exist_ok=False)

for path in train_files:
    os.symlink(path.resolve(), smoke_dataset_dir / path.name)
tokenizer_files = sorted(source_tokenizer_dir.glob('fineweb_8192_bpe.*'))
if not tokenizer_files:
    raise SystemExit(f'expected tokenizer files in {source_tokenizer_dir}, found 0 matching fineweb_8192_bpe.*')
for path in tokenizer_files:
    os.symlink(path.resolve(), smoke_tokenizer_dir / path.name)

smoke_tokens = np.concatenate(val_parts)
if smoke_tokens.size != SMOKE_VAL_MAX_TOKENS:
    raise SystemExit(
        f'constructed capped validation token count mismatch: expected {SMOKE_VAL_MAX_TOKENS}, found {smoke_tokens.size}'
    )
smoke_header = first_header.copy()
smoke_header[2] = SMOKE_VAL_MAX_TOKENS
smoke_val_path = smoke_dataset_dir / 'fineweb_val_000000.bin'
with smoke_val_path.open('wb') as f:
    f.write(smoke_header.astype('<i4', copy=False).tobytes())
    f.write(smoke_tokens.astype(TOKEN_DTYPE, copy=False).tobytes())

print(f'Source data: {len(train_files)} train shard(s), {len(val_files)} val shard(s), full_val_tokens={full_val_tokens}')
print(
    'Smoke data ready: '
    f'train_shards={len(train_files)} '
    f'smoke_val_tokens={SMOKE_VAL_MAX_TOKENS} '
    f'eval_sequences={SMOKE_VAL_MAX_TOKENS // EVAL_SEQ_LEN} '
    f'ttt_chunks={SMOKE_VAL_MAX_TOKENS // TTT_CHUNK_TOKENS} '
    f'ratio={SMOKE_VAL_MAX_TOKENS / full_val_tokens:.4f} '
    f'data_dir={smoke_data_dir}'
)
PY

## April 9 — `SP8192_3LayerRecur_ParResid_QK525_LegalTTT`

Baseline record. Uses legacy fixed-bits export (6-bit matrices, 8-bit embeddings).
No attention gate, no multi-phase TTT.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`, `artifact bytes`.

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
REC="$REPO/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/data_smoke" \
SEED=42 \
RUN_ID=smoke_apr9 \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

## April 20 #1 — `SP8192_AttnGate_MultiPhaseTTT_LaCT`

Adds zero-init attention output gate (`ATTN_GATE_ENABLED=1`) and 4-phase score-first TTT
(`MULTIPHASE_TTT_ENABLED=1`). LaCT adapter is disabled here to keep the run fast.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: `pre-quantization post-ema`, `quantized`, `quantized_sliding_window`,
`allocator_selected`, `artifact bytes`.

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
REC="$REPO/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/data_smoke" \
SEED=42 \
RUN_ID=smoke_apr20_attngate_mpttt \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

## April 20 #2 — `SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT`

Combines 3-layer recurrence + parallel residual with attention output gate
(`ATTN_OUT_GATE_ENABLED=1`, note different flag name from #1) and 4-phase TTT
(`TTT_PHASES=4`, the default). Has built-in torch SDPA fallback — no flash-attn needed.
Uses the entropy-constrained GPTQ allocator.

Expected log lines: same as April 20 #1.

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
COLAB_DIR="$REPO/colab/2026-04-23_SmokeTest_Records"
REC="$REPO/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT"
cd "$REC"
mkdir -p logs

PYTHONPATH="$COLAB_DIR:${PYTHONPATH:-}" \
DATA_DIR="$REPO/data_smoke" \
SEED=42 \
RUN_ID=smoke_apr20_mp4ttt \
ITERATIONS=300 \
TRAIN_BATCH_TOKENS=32768 \
VAL_BATCH_TOKENS=32768 \
MAX_WALLCLOCK_SECONDS=0 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=50 \
PG_COLAB_DISABLE_COMPILE=1 \
GPTQ_CALIBRATION_BATCHES=8 \
GPTQ_RESERVE_SECONDS=30 \
python3 train_gpt.py

## Compare final metrics from all three capped-val runs

Run this after all three training cells finish. `val_tokens` should match the explicit smoke cap in every log.

In [ ]:
%%bash
set -euo pipefail
REPO="$(pwd)/parameter-golf"
APR9="$REPO/records/track_10min_16mb/2026-04-09_SP8192_3LayerRecur_ParResid_QK525_LegalTTT/logs/smoke_apr9.txt"
APR20A="$REPO/records/track_10min_16mb/2026-04-20_SP8192_AttnGate_MultiPhaseTTT_LaCT/logs/smoke_apr20_attngate_mpttt.txt"
APR20B="$REPO/records/track_10min_16mb/2026-04-20_SP8192_3LayerRecur_ParResid_QK525_AttnGate_MP4TTT/logs/smoke_apr20_mp4ttt.txt"

for LOG in "$APR9" "$APR20A" "$APR20B"; do
  echo "=== $(basename $(dirname $LOG))/$(basename $LOG) ==="
  grep -E "val_tokens:|pre-quantization post-ema|quantized|artifact bytes|allocator_selected" "$LOG" | tail -20
  echo
done